# modular-diffusion: the time split

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.3.1/colab/time_split.ipynb)

What a stage of the noise level costs when it shares nothing, against one model over the whole range (`geolip.modular_diffusion.experiments.time_split`). Three cells, run one at a time on a GPU runtime:

1. install at a release tag and print the environment;
2. the data with its content hash, a smoke run whose picture score is checked against a second implementation, the steps per second of 1, 3 and 6 jobs at once, and the size of the whole job list;
3. the job list, run by several runners that share it; each finished job is shipped to the Hub at once; a progress line a minute; the summary; then the machine is given back.

A stopped session resumes by running the cells again: finished jobs keep their markers under `OUT`, and on a new machine the jobs already on the Hub are fetched back first.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data,hub,judge,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.3.1"
import os, sys, time, json, shutil, threading, subprocess
import torch, geolip.modular_diffusion as MD
from geolip.modular_diffusion import environment
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 3, 1), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
environment.check(None)
print("card:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)", "| cpus:", os.cpu_count())
OUT = "/content/time_split"          # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The data comes from the Hub as parquet files (one train and one test file) and becomes one npz of uint8 arrays; its content hash does not depend on the zip container. The smoke run trains `plain` at 1% of its steps and scores 5,000 pictures; the same pictures are then scored with torchmetrics' FrechetInceptionDistance twice: with the same resize as the library's judge (the two numbers should agree closely: the same feature network, another square-root routine) and with torchmetrics' default resize (a different number: small pictures are enlarged another way). The throughput measure runs 600 steps alone, three at once and six at once, and reads each run's own training seconds. The last lines print the job list's size and a time estimate.

In [ ]:
# C2 - the data, a smoke run with the picture score checked twice, the throughput of 1, 3 and 6 jobs at once, the job list's size
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data, score, linker
from geolip.modular_diffusion.beds import PicturesBed
from geolip.modular_diffusion.experiments import time_split
NPZ = f"{OUT}/data/cifar10_32.npz"
REF = f"{OUT}/data/reference_inception.npz"     # the reference statistics: computed once here, read by every job
if not os.path.exists(NPZ):
    snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
    data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=NPZ)
EXPECTED_CONTENT_HASH = "be50ca799cbe6795ff13bca750f53f16c974b9f4903082fc06031b9719700016"
got = data.npz_content_hash(NPZ)
print("content hash", got, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or got == EXPECTED_CONTENT_HASH, "the npz is not the expected data"

SMOKE = f"{OUT}/smoke/plain"
smoke = time_split.main(variant="plain", seed=0, data=NPZ, out=SMOKE, device="cuda", scale=0.01, score=1, score_n=5000, ref=REF)
assert "score" in smoke["score"], smoke["score"]
from torchmetrics.image.fid import FrechetInceptionDistance
bed = PicturesBed(NPZ, "cuda")
model = linker.link(f"{SMOKE}/model", device="cuda")
fake = score.sample_pictures(model, bed, 5000, steps=50, batch=1000, device="cuda")
real = bed.train_pictures()
others = {}
for aa in (False, True):     # False: the same resize as the library's judge; True: torchmetrics' own default resize
    fid = FrechetInceptionDistance(feature=2048, antialias=aa).to("cuda")
    for i in range(0, real.shape[0], 1000):
        fid.update(score.to_pictures(real[i:i + 1000], bed).to("cuda"), real=True)
    for i in range(0, fake.shape[0], 1000):
        fid.update(fake[i:i + 1000].to("cuda"), real=False)
    others[aa] = float(fid.compute())
    del fid
mine = smoke["score"]["score"]
print(f"picture score: library {mine:.4f} | torchmetrics, the same resize {others[False]:.4f} (relative difference "
      f"{abs(mine - others[False]) / others[False]:.2e}) | torchmetrics, its default resize {others[True]:.4f} (5,000 pictures)")
del model, bed, real, fake
torch.cuda.empty_cache()

def timed(n):
    """n runs of 600 steps started at once (separate processes, as the runners start them) -> steps per second in all."""
    folders = [f"{OUT}/throughput/n{n}_{k}" for k in range(n)]
    procs = [subprocess.Popen([sys.executable, "-m", "geolip.modular_diffusion.run", "time_split", "variant=plain", f"seed={k}",
                               f"data={NPZ}", f"out={f}", "device=cuda", "scale=0.1", "sheet=0", "heldout=256", "threads=2"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT) for k, f in enumerate(folders)]
    assert all(p.wait() == 0 for p in procs), "a throughput run failed"
    return sum(600 / json.load(open(f"{f}/result.json"))["train_seconds"] for f in folders)
rates = {n: timed(n) for n in (1, 3, 6)}
print("steps per second in all:", {n: round(r, 1) for n, r in rates.items()})
RUNNERS = 1
for n in (3, 6):
    if rates[n] > 1.15 * rates[RUNNERS]:     # more runners only for a gain above 15%
        RUNNERS = n
print("runners to start:", RUNNERS)

# the job list: (tag, variant, seed), the runs that others start from first; every variant at seeds 0 and 1, the
# reference also at seeds 2 to 5 (its seed spread on this card) and seed 0 a second time (the card's own repeat)
FIRST = ["plain_d4", "plain_4k", "plain"]
REST = [v for v in time_split.VARIANTS if v not in FIRST]
JOBS = [(f"{v}_s0", v, 0) for v in FIRST] + [("plain_s0_again", "plain", 0)] + [(f"{v}_s1", v, 1) for v in FIRST]
JOBS += [(f"plain_s{s}", "plain", s) for s in (2, 3, 4, 5)] + [(f"{v}_s{s}", v, s) for s in (0, 1) for v in REST]
COST = {"plain_wide": 2.5, "plain_d4": 0.6, "tails_joint": 1.7, "tails_mixed": 1.3}   # a step's cost against the reference model's
STEPS = {tag: sum(n for _, n in time_split.plan(v, 1.0)) * COST.get(v, 1.0) for tag, v, _ in JOBS}

# the reads on the saved models (no training): (tag, arguments, the jobs it waits for, sampling steps); {R} = the runs folder
READS = []
for s in (0, 1):
    o = 1 - s
    for d in ("split", "split_long", "split_soft"):      # one stage of the plain model served by a separately trained body
        READS += [(f"swap_{d}_k{k}_s{s}", f"variant=swap base={{R}}/plain_s{s} donor={{R}}/{d}_s{s} stage={k}", [f"plain_s{s}", f"{d}_s{s}"], 50) for k in (0, 1, 2)]
    for d in ("split", "split_long"):                    # one stage's body from the other seed's run
        READS += [(f"twin_{d}_k{k}_s{s}", f"variant=twin base={{R}}/{d}_s{s} donor={{R}}/{d}_s{o} stage={k}", [f"{d}_s{s}", f"{d}_s{o}"], 50) for k in (0, 1, 2)]
    for d in ("split", "split_long", "tails_joint"):     # the clusters served to the wrong stages, scored
        READS += [(f"wrong_{d}_s{s}", f"variant=wrong base={{R}}/{d}_s{s}", [f"{d}_s{s}"], 50)]
    for d in ("split", "split_long", "split_soft", "tails_joint", "tails_alone", "copies"):   # the blend width at reading
        READS += [(f"blend_{d}_w{w}_s{s}", f"variant=blend base={{R}}/{d}_s{s} soft={w}", [f"{d}_s{s}"], 50) for w in ("0", "0.12")]
    for d in ("plain", "split"):                         # sampling steps shared among the stages
        READS += [(f"steps_{d}_uniform_n50_s{s}", f"variant=steps base={{R}}/{d}_s{s} plan=uniform n=50", [f"{d}_s{s}"], 50)]
        READS += [(f"steps_{d}_{p}_n{n}_s{s}", f"variant=steps base={{R}}/{d}_s{s} plan={p} n={n}", [f"{d}_s{s}"], n)
                  for n in (10, 20, 50) for p in ("width", "start", "middle", "finish")]
for d in ("plain", "split"):                             # the score's own sampling spread: three more noise seeds, seed 0
    READS += [(f"spread_{d}_q{q}_s0", f"variant=steps base={{R}}/{d}_s0 plan=uniform n=50 score_seed={q}", [f"{d}_s0"], 50) for q in (1, 2, 3)]
STEPS.update({tag: 360 + 11 * n for tag, _, _, n in READS})   # a read's time in reference-model steps (loading, sampling, scoring)
total = sum(STEPS.values())
print(f"the job list: {len(JOBS)} training jobs and {len(READS)} reads, {total:,.0f} reference-model steps in all; about "
      f"{total / rates[RUNNERS] / 60:.0f} min with {RUNNERS} runner(s), plus about {len(JOBS) * 1.5:.0f} min of the jobs' own reads and scoring")

The roster holds one line per job; `needs=` holds a job until the run it starts from is DONE. After the training jobs come the reads on their saved models (`geolip.modular_diffusion.experiments.chain_reads`: no training; one stage swapped, a stage from the other seed, the wrong assembly, the blend width, the step plans), each waiting for the jobs it reads. The runners share the roster by claims. Each finished job folder is shipped to `REPO` under `PREFIX` with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). With `REPO = ""` nothing is shipped and the machine is kept. When the roster has ended and its report is shipped, the machine is given back (`RELEASE`), since an idle machine keeps spending.

In [ ]:
# C3 - the roster, the runners, shipping after each job, a progress line a minute, the summary, the machine given back
from geolip.modular_diffusion import jobs, hub
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the Hub repo the results go to ("" = not shipped)
PREFIX = "modular_diffusion/time_split"
RELEASE = True                              # give the machine back when the roster has ended and the report is shipped
RUNS = f"{OUT}/runs"
NEEDS = {"tails_alone": ("stem_from", "plain_d4"), "copies_4k": ("source", "plain_4k"), "copies": ("source", "plain"),
         "copies_soft": ("source", "plain")}
THREADS = max(2, min(4, (os.cpu_count() or 4) // RUNNERS))
TAIL = f"data={NPZ} device=cuda score=1 ref={REF} threads={THREADS}"
EXTRA_JOB, EXTRA_READ = "", ""      # more arguments for every training job / every read (empty in a real run)
lines = []
for tag, v, seed in JOBS:
    args = f"time_split variant={v} seed={seed} {TAIL}{EXTRA_JOB}"
    needs = ""
    if v in NEEDS:
        key, src = NEEDS[v]
        args += f" {key}={RUNS}/{src}_s{seed}"
        needs = f" | needs={src}_s{seed}"
    lines.append(f"{tag} | 60 | geolip.modular_diffusion.run | {args}{needs}")
for tag, args, needs, _ in READS:
    where, wait = args.replace("{R}", RUNS), ",".join(needs)
    lines.append(f"{tag} | 30 | geolip.modular_diffusion.run | chain_reads {where} {TAIL}{EXTRA_READ} | needs={wait}")
ROSTER = f"{OUT}/roster.txt"
with open(ROSTER, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(lines) + "\nEND\n")
tags = [tag for tag, _, _ in JOBS] + [tag for tag, _, _, _ in READS]
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:    # on a new machine: fetch the jobs already shipped and mark them done
        got_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/*"], local_dir=f"{OUT}/hub")
        for t in tags:
            src = f"{got_dir}/{PREFIX}/{t}"
            if os.path.exists(f"{src}/result.json") and not mark(t, "DONE"):
                shutil.copytree(src, f"{RUNS}/{t}", dirs_exist_ok=True)
                open(f"{RUNS}/{t}/DONE", "w").write("fetched back from the Hub\n")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")

def after_job(tag, folder, ok):
    if ok and REPO:
        hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"time split: {tag}")

runners = [threading.Thread(target=jobs.run_roster, daemon=True,
                            kwargs=dict(roster=ROSTER, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster_{k}.log", card_room_mib=None,
                                        ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job,
                                        env={"OMP_NUM_THREADS": str(THREADS), "MKL_NUM_THREADS": str(THREADS)}))
           for k in range(RUNNERS)]
t0 = time.time()
left0 = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; {left0:,.0f} steps to run with {RUNNERS} runner(s)")
for th in runners:
    th.start()
while any(th.is_alive() for th in runners):
    time.sleep(60)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    eta = spent * left / (left0 - left) if left0 > left else float("nan")
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {eta:.0f} min left", flush=True)

from geolip.modular_diffusion.experiments import chain_reads
summary = time_split.summarize(RUNS) + "\n" + chain_reads.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
for name in ["roster.txt"] + [f"roster_{k}.log" for k in range(RUNNERS)]:
    if os.path.exists(f"{OUT}/{name}"):
        shutil.copy(f"{OUT}/{name}", f"{REPORT}/{name.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed (a job's own log files are not shipped with its folder)
    if mark(t, "FAILED"):
        for name in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{name}"):
                shutil.copy(f"{RUNS}/{t}/{name}", f"{REPORT}/failed_{t}_{name.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "time split: summary and roster logs") if REPO else None
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()